# Session 3: train a neural network and watch backpropagation
**A. Real-world problem.** One line cannot separate two interleaved spirals (or XOR). Stack
neurons in layers and the dividing curve can bend. But how do we find 100 weights? By
*backpropagation*: the chain rule, applied layer by layer, to get the gradient of the loss
for every weight at once. If one formula is wrong the network silently learns badly, so we
test every gradient against finite differences.

**Plan.** (1) forward pass, (2) backward pass (derived in the README, section B), (3) gradient
check, (4) train on XOR, moons and spirals, (5) optimisers, (6) overfitting, initialisation,
vanishing gradients and a learning rate that is too high.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: data made in code (seeded, so everyone gets the same numbers)
Rows are samples. `y` is 0 or 1.

In [ ]:
XOR_X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
XOR_Y = np.array([0.0, 1.0, 1.0, 0.0])


def make_moons(n, noise, seed):
    """Two interleaving half circles, n points in total, Gaussian noise of standard deviation `noise`."""
    rng = np.random.default_rng(seed)
    h = n // 2
    t = rng.uniform(0, np.pi, h)
    t2 = rng.uniform(0, np.pi, n - h)
    a = np.c_[np.cos(t), np.sin(t)]
    b = np.c_[1 - np.cos(t2), 0.5 - np.sin(t2)]
    X = np.r_[a, b] + rng.normal(0, noise, (n, 2))
    y = np.r_[np.zeros(h), np.ones(n - h)]
    order = rng.permutation(n)                            # shuffle so that slices are mixed
    return X[order], y[order]


def make_spirals(n, noise, seed, turns=1.5):
    """Two spiral arms (n points in total), the second arm is the first turned by 180 degrees."""
    rng = np.random.default_rng(seed)
    h = n // 2
    r = np.sqrt(rng.uniform(0.02, 1.0, h))               # radius 0.14 .. 1
    th = r * turns * 2 * np.pi
    arm = np.c_[r * np.cos(th), r * np.sin(th)]
    X = np.r_[arm, -arm] + rng.normal(0, noise, (2 * h, 2))
    y = np.r_[np.zeros(h), np.ones(h)]
    order = rng.permutation(2 * h)
    return X[order], y[order]


def split(X, y, test_fraction, seed):
    """Shuffle with a seed and cut into (X_train, y_train, X_test, y_test)."""
    idx = np.random.default_rng(seed).permutation(len(y))
    k = int(round(len(y) * (1 - test_fraction)))
    return X[idx[:k]], y[idx[:k]], X[idx[k:]], y[idx[k:]]

## Step 2: activations
Each hidden neuron applies a bend `phi` to its weighted sum. We store the derivative too.

In [ ]:
ACT = {
    "tanh": (np.tanh, lambda z: 1.0 - np.tanh(z) ** 2),
    "relu": (lambda z: np.maximum(z, 0.0), lambda z: (z > 0).astype(float)),
    "sigmoid": (lambda z: 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60))),
                lambda z: (1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))) * (1 - 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60))))),
}


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))

## Step 3: parameters, forward pass, loss
`params` is a list of `[W, b]` pairs, one per layer. For a two-layer network
(2 inputs -> h hidden -> 1 output): `Z1 = X W1 + b1`, `H = phi(Z1)`, `Z2 = H W2 + b2`,
`P = sigmoid(Z2)`, loss `L = -mean(y log P + (1 - y) log(1 - P))`.

In [ ]:
def init_params(sizes, seed, scheme="xavier"):
    """sizes e.g. [2, 8, 1]. scheme: 'xavier' (std sqrt(1/fan_in)), 'he' (std sqrt(2/fan_in)),
    'zeros', or 'const' (every weight 0.5, a symmetric start)."""
    rng = np.random.default_rng(seed)
    params = []
    for fi, fo in zip(sizes[:-1], sizes[1:]):
        if scheme == "zeros":
            W = np.zeros((fi, fo))
        elif scheme == "const":
            W = np.full((fi, fo), 0.5)
        elif scheme == "he":
            W = rng.normal(0, np.sqrt(2.0 / fi), (fi, fo))
        else:
            W = rng.normal(0, np.sqrt(1.0 / fi), (fi, fo))
        params.append([W, np.zeros(fo)])
    return params


def forward(params, X, act="tanh"):
    """Returns (P, cache). cache = list of (layer input A_prev, pre-activation Z) per layer."""
    phi = ACT[act][0]
    A, cache = X, []
    for k, (W, b) in enumerate(params):
        Z = A @ W + b
        cache.append((A, Z))
        A = sigmoid(Z) if k == len(params) - 1 else phi(Z)
    return A.ravel(), cache


def bce(P, y):
    P = np.clip(P, 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(P) + (1 - y) * np.log(1 - P)))


def loss_of(params, X, y, act="tanh"):
    return bce(forward(params, X, act)[0], y)

## Step 4: backpropagation
Output layer: `dZ_last = (P - y) / N` (the same "prediction minus truth" as Session 2).
For each layer, from the last to the first:
    dW = A_prev^T dZ        db = column sums of dZ
    dA_prev = dZ W^T        dZ_prev = dA_prev * phi'(Z_prev)
For two layers this reads: `dL/dW2 = H^T dZ2`, `dL/dW1 = X^T (dZ2 W2^T * phi'(Z1))`.

In [ ]:
def backward(params, cache, P, y, act="tanh"):
    """Gradients [[dW, db], ...] with the same shapes as params."""
    dphi = ACT[act][1]
    N = len(y)
    dZ = (P - y).reshape(-1, 1) / N
    grads = [None] * len(params)
    for k in range(len(params) - 1, -1, -1):
        A_prev, _ = cache[k]
        W = params[k][0]
        grads[k] = [A_prev.T @ dZ, dZ.sum(axis=0)]
        if k > 0:
            dZ = (dZ @ W.T) * dphi(cache[k - 1][1])
    return grads


def loss_and_grads(params, X, y, act="tanh"):
    P, cache = forward(params, X, act)
    return bce(P, y), backward(params, cache, P, y, act)


def grad_check(params, X, y, act="tanh", h=1e-5):
    """Largest relative error |a - n| / max(|a| + |n|, 1e-12) between the backprop gradient a and
    the central finite difference n, over every single weight and bias."""
    _, grads = loss_and_grads(params, X, y, act)
    worst = 0.0
    for k, (W, b) in enumerate(params):
        for arr, g in ((W, grads[k][0]), (b, grads[k][1])):
            it = np.nditer(arr, flags=["multi_index"])
            for _ in it:
                i = it.multi_index
                old = arr[i]
                arr[i] = old + h
                lp = loss_of(params, X, y, act)
                arr[i] = old - h
                lm = loss_of(params, X, y, act)
                arr[i] = old
                num = (lp - lm) / (2 * h)
                worst = max(worst, abs(g[i] - num) / max(abs(g[i]) + abs(num), 1e-12))
    return worst

## Step 5: optimisers and the training loop
Plain SGD: `w = w - lr * g`. Momentum: `v = beta v + g; w = w - lr * v`.
Adam: running averages of g and g^2, with bias correction:
    m = b1 m + (1 - b1) g;  v = b2 v + (1 - b2) g^2
    m_hat = m / (1 - b1^t);  v_hat = v / (1 - b2^t);  w = w - lr * m_hat / (sqrt(v_hat) + eps)

In [ ]:
class Optimiser:
    """kind: 'sgd', 'momentum' or 'adam'. Updates params in place."""

    def __init__(self, params, kind="adam", lr=0.01, beta=0.9, b1=0.9, b2=0.999, eps=1e-8):
        self.kind, self.lr, self.beta, self.b1, self.b2, self.eps, self.t = kind, lr, beta, b1, b2, eps, 0
        self.m = [[np.zeros_like(W), np.zeros_like(b)] for W, b in params]
        self.v = [[np.zeros_like(W), np.zeros_like(b)] for W, b in params]

    def step(self, params, grads):
        self.t += 1
        for k in range(len(params)):
            for j in range(2):
                g, w = grads[k][j], params[k][j]
                if self.kind == "sgd":
                    w -= self.lr * g
                elif self.kind == "momentum":
                    self.m[k][j] = self.beta * self.m[k][j] + g
                    w -= self.lr * self.m[k][j]
                else:
                    self.m[k][j] = self.b1 * self.m[k][j] + (1 - self.b1) * g
                    self.v[k][j] = self.b2 * self.v[k][j] + (1 - self.b2) * g * g
                    mh = self.m[k][j] / (1 - self.b1 ** self.t)
                    vh = self.v[k][j] / (1 - self.b2 ** self.t)
                    w -= self.lr * mh / (np.sqrt(vh) + self.eps)


def predict(params, X, act="tanh"):
    return (forward(params, X, act)[0] > 0.5).astype(int)


def accuracy(params, X, y, act="tanh"):
    return float(np.mean(predict(params, X, act) == y))


def train(params, X, y, act="tanh", kind="adam", lr=0.01, epochs=200, batch=None, seed=0, Xt=None, yt=None, snapshot=None):
    """Minibatch training (batch=None means the whole data set). Returns a history dict with the
    loss and accuracy after every epoch (and test accuracy if Xt, yt are given). snapshot(epoch, params)
    is called after every epoch if given."""
    rng = np.random.default_rng(seed)
    opt = Optimiser(params, kind, lr)
    N = len(y)
    bs = N if batch is None else batch
    hist = {"loss": [], "acc": [], "test_acc": []}
    for ep in range(epochs):
        idx = rng.permutation(N)
        for s in range(0, N, bs):
            sel = idx[s:s + bs]
            P, cache = forward(params, X[sel], act)
            opt.step(params, backward(params, cache, P, y[sel], act))
        hist["loss"].append(loss_of(params, X, y, act))
        hist["acc"].append(accuracy(params, X, y, act))
        if Xt is not None:
            hist["test_acc"].append(accuracy(params, Xt, yt, act))
        if snapshot is not None:
            snapshot(ep, params)
    return hist


def confusion(y_true, y_pred):
    """[[true 0 predicted 0, true 0 predicted 1], [true 1 predicted 0, true 1 predicted 1]]."""
    m = np.zeros((2, 2), dtype=int)
    for t, p in zip(y_true.astype(int), y_pred.astype(int)):
        m[t, p] += 1
    return m

## Step 6: gradients layer by layer in a deep network
With a sigmoid, `phi'(z) <= 0.25`. Each layer the gradient passes through multiplies it by
`phi'` and by the weights, so in a deep sigmoid network the gradient reaching the first layers
can be tiny. We measure the norm of dL/dW in each layer.

In [ ]:
def layer_grad_norms(depth, width, act, scheme, X, y, seed=0):
    """Frobenius norm of dL/dW for each of the depth + 1 weight matrices (first layer first)."""
    params = init_params([X.shape[1]] + [width] * depth + [1], seed, scheme)
    _, grads = loss_and_grads(params, X, y, act)
    return [float(np.linalg.norm(g[0])) for g in grads]


def grid_predict(params, act, lim=1.6, n=120):
    gx, gy = np.meshgrid(np.linspace(-lim, lim, n), np.linspace(-lim, lim, n))
    P = forward(params, np.c_[gx.ravel(), gy.ravel()], act)[0].reshape(gx.shape)
    return gx, gy, P

## Step 7: run the experiments

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)

    # gradient check
    p = init_params([2, 6, 1], 0)
    err = grad_check(p, *make_moons(40, 0.2, 0))
    print(f"gradient check (tanh, 2-6-1): worst relative error {err:.1e}")

    # XOR
    p = init_params([2, 4, 1], 1)
    train(p, XOR_X, XOR_Y, lr=0.05, epochs=300)
    print("XOR predictions:", predict(p, XOR_X), "accuracy", accuracy(p, XOR_X, XOR_Y))

    # spirals
    X, y = make_spirals(400, 0.05, 0)
    Xtr, ytr, Xte, yte = split(X, y, 0.25, 0)
    p = init_params([2, 64, 1], 0)
    h = train(p, Xtr, ytr, lr=0.02, epochs=600, batch=32, seed=0, Xt=Xte, yt=yte)
    print(f"spirals: train accuracy {h['acc'][-1]:.3f}, test accuracy {h['test_acc'][-1]:.3f}")
    gx, gy, P = grid_predict(p, "tanh")
    fig, axs = plt.subplots(1, 2, figsize=(9, 4))
    axs[0].contourf(gx, gy, P, levels=[0, 0.5, 1], alpha=0.25, colors=["tab:orange", "tab:blue"])
    axs[0].scatter(*Xtr[ytr == 0].T, c="tab:orange", s=12)
    axs[0].scatter(*Xtr[ytr == 1].T, c="tab:blue", s=12)
    axs[0].set_xlabel("input x1")
    axs[0].set_ylabel("input x2")
    axs[0].set_title("Decision boundary on two spirals")
    axs[1].plot(h["loss"], label="training loss")
    axs[1].set_xlabel("epoch")
    axs[1].set_ylabel("cross-entropy loss (no unit)")
    ax2 = axs[1].twinx()
    ax2.plot(h["test_acc"], color="tab:green", label="test accuracy")
    ax2.set_ylabel("test accuracy (fraction)")
    axs[1].set_title("Loss falls, accuracy rises")
    axs[1].grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "spiral_boundary.png", dpi=120)

    # optimisers
    Xm, ym = make_moons(300, 0.2, 1)
    fig2, ax3 = plt.subplots(figsize=(6, 4))
    for kind, lr in (("sgd", 0.05), ("momentum", 0.05), ("adam", 0.05)):
        pp = init_params([2, 16, 1], 2)
        hh = train(pp, Xm, ym, kind=kind, lr=lr, epochs=40, batch=32, seed=3)
        ax3.plot(hh["loss"], label=f"{kind} (lr {lr})")
        print(f"{kind}: loss after 40 epochs {hh['loss'][-1]:.3f}")
    ax3.set_xlabel("epoch")
    ax3.set_ylabel("training loss (no unit)")
    ax3.set_title("Minibatch training on moons: SGD, momentum, Adam")
    ax3.grid(alpha=0.3)
    ax3.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "optimisers.png", dpi=120)

    # overfitting
    Xo, yo = make_moons(1040, 0.35, 5)
    Xs, ys, Xte2, yte2 = Xo[:40], yo[:40], Xo[40:], yo[40:]
    fig3, axs3 = plt.subplots(1, 2, figsize=(9, 4))
    for ax, hid in zip(axs3, (3, 100)):
        pp = init_params([2, hid, 1], 4)
        hh = train(pp, Xs, ys, lr=0.01, epochs=1500, Xt=Xte2, yt=yte2)
        ax.plot(hh["acc"], label="training accuracy")
        ax.plot(hh["test_acc"], label="test accuracy")
        ax.set_title(f"{hid} hidden neurons, 40 noisy points")
        ax.set_xlabel("epoch")
        ax.set_ylabel("accuracy (fraction)")
        ax.set_ylim(0.4, 1.02)
        ax.grid(alpha=0.3)
        ax.legend(loc="lower right")
        print(f"{hid} hidden: train {hh['acc'][-1]:.3f}, test {hh['test_acc'][-1]:.3f}")
    fig3.tight_layout()
    fig3.savefig(OUT / "overfitting.png", dpi=120)

    # vanishing gradients
    Xv, yv = make_moons(200, 0.2, 7)
    fig4, ax4 = plt.subplots(figsize=(6.5, 4))
    for act, scheme in (("sigmoid", "xavier"), ("tanh", "xavier"), ("relu", "he")):
        gn = layer_grad_norms(10, 16, act, scheme, Xv, yv)
        ax4.semilogy(range(1, len(gn) + 1), gn, "o-", label=f"{act} ({scheme})")
        print(f"{act}: gradient norm layer 1 = {gn[0]:.2e}, last layer = {gn[-1]:.2e}")
    ax4.set_xlabel("layer (1 = nearest the input)")
    ax4.set_ylabel("norm of dL/dW (log scale)")
    ax4.set_title("Gradient norm per layer in a 10-hidden-layer network")
    ax4.grid(alpha=0.3, which="both")
    ax4.legend()
    fig4.tight_layout()
    fig4.savefig(OUT / "vanishing_gradients.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change the hidden activation in `train` to "relu" with the "he" initialisation and compare the loss after 40 epochs on the moons data with the tanh value printed by the lesson.
2. **Stretch.** Add a deliberate bug to `backward` (for example drop the division by N) and run `grad_check`. Describe how the error changes, then remove the bug.
3. **Explain.** Explain why, with all weights zero, the two hidden neurons of a network always receive exactly the same gradient, and how random initial weights break the tie.

## Assessment
- **A. Conceptual.** Why does the gradient check use two-sided differences and a relative error, and what does it tell you if the error is 1e-2 instead of 1e-9?
- **B. Numerical.** An output neuron gives P = 0.8 for a point with label 1 and the hidden activations are H = (0.5, -0.5), with N = 1. Find dZ2, dL/dW2 and the loss.
- **C. Graph and simulation interpretation.** In the gradient-norm figure, what is the ratio of the first-layer to the last-layer gradient norm for a 10-layer sigmoid network, and for tanh and ReLU? What does it say about training the first layers?
- **D. Design and optimisation challenge.** Design (mini-project): on the two-moons data (400 points, noise 0.25, seed 11, 25 percent test) choose the smallest number of hidden neurons among 2, 4, 16 that reaches 0.92 test accuracy, and say what is lost by making the network larger than needed.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []

    # --- worked example: a 2-2-1 tanh network, one sample, by hand ---
    W1 = np.array([[0.5, -0.3], [0.2, 0.4]])
    W2 = np.array([[0.6], [-0.4]])
    params = [[W1.copy(), np.zeros(2)], [W2.copy(), np.zeros(1)]]
    x = np.array([[1.0, 0.0]])
    y1 = np.array([1.0])
    P, cache = L.forward(params, x)
    H = np.tanh(cache[0][1])
    checks.append(("worked example: Z1 = (0.5, -0.3), H = (0.4621, -0.2913)",
                   np.allclose(cache[0][1], [[0.5, -0.3]]) and np.allclose(H, [[0.4621, -0.2913]], atol=1e-4)))
    checks.append(("worked example: Z2 = 0.3938, P = 0.5972, loss = -ln P = 0.5155",
                   abs(cache[1][1][0, 0] - 0.3938) < 1e-4 and abs(P[0] - 0.5972) < 1e-4 and abs(L.bce(P, y1) - 0.5155) < 1e-4))
    g = L.backward(params, cache, P, y1)
    checks.append(("worked example: dZ2 = P - y = -0.4028 and dL/dW2 = (-0.1861, 0.1173)",
                   abs(g[1][1][0] + 0.4028) < 1e-4 and np.allclose(g[1][0].ravel(), [-0.1861, 0.1173], atol=1e-4)))
    checks.append(("worked example: dL/dW1 first row = (-0.1901, 0.1474), second row = 0 (because x2 = 0)",
                   np.allclose(g[0][0][0], [-0.1901, 0.1474], atol=1e-4) and np.allclose(g[0][0][1], 0.0)))
    # hand chain rule for the first weight, written out
    dH = (P[0] - 1.0) * W2.ravel()
    dz1 = dH * (1 - H.ravel() ** 2)
    checks.append(("chain rule by hand: dL/dH = (-0.2417, 0.1611), tanh slope (0.7865, 0.9151)",
                   np.allclose(dH, [-0.2417, 0.1611], atol=1e-4) and np.allclose(1 - H.ravel() ** 2, [0.7865, 0.9151], atol=1e-4)
                   and np.allclose(dz1, g[0][1], atol=1e-12)))
    err0 = L.grad_check(params, x, y1)
    checks.append((f"worked example passes the gradient check (relative error {err0:.1e} < 1e-6)", err0 < 1e-6))

    Hh = np.array([0.5, -0.5])
    checks.append(("hand calculation: P = 0.8, y = 1, H = (0.5, -0.5) gives dZ2 = -0.2, dL/dW2 = (-0.1, 0.1) and loss 0.2231",
                   np.allclose((0.8 - 1.0) * Hh, [-0.1, 0.1]) and abs(L.bce(np.array([0.8]), np.array([1.0])) - 0.2231) < 1e-4))

    # --- the key correctness test ---
    Xg, yg = L.make_moons(40, 0.2, 0)
    errs = {}
    for act in ("tanh", "sigmoid", "relu"):
        errs[act] = L.grad_check(L.init_params([2, 6, 1], 0), Xg, yg, act)
    checks.append(("gradient check, two layers 2-6-1, tanh / sigmoid / relu: worst relative error below 1e-6 (about 1e-9)",
                   max(errs.values()) < 1e-6))
    err3 = L.grad_check(L.init_params([2, 5, 4, 1], 3), Xg, yg, "tanh")
    checks.append((f"gradient check also passes for three layers 2-5-4-1 ({err3:.1e} < 1e-6)", err3 < 1e-6))
    saved = L.ACT["tanh"]
    L.ACT["tanh"] = (np.tanh, lambda z: np.ones_like(z))          # a deliberate bug: forget the slope phi'
    bad = L.grad_check(L.init_params([2, 6, 1], 0), Xg, yg, "tanh")
    L.ACT["tanh"] = saved
    checks.append((f"the check catches a bug: dropping phi' gives relative error {bad:.2f} (> 1e-2)", bad > 1e-2))

    # --- Adam by hand ---
    pa = [[np.array([[1.0]]), np.array([0.0])]]
    opt = L.Optimiser(pa, "adam", lr=0.1)
    opt.step(pa, [[np.array([[2.0]]), np.array([0.0])]])
    checks.append(("hand calculation: the first Adam step with gradient 2 moves the weight by lr = 0.1 (m_hat = 2, sqrt(v_hat) = 2)",
                   abs(pa[0][0][0, 0] - 0.9) < 1e-6))
    pm = [[np.array([[1.0]]), np.array([0.0])]]
    om = L.Optimiser(pm, "momentum", lr=0.1, beta=0.9)
    gm = [[np.array([[1.0]]), np.array([0.0])]]
    om.step(pm, gm)
    om.step(pm, gm)
    checks.append(("hand calculation: momentum with constant gradient 1: steps of 0.1 then 0.19, so w = 1 - 0.29",
                   abs(pm[0][0][0, 0] - 0.71) < 1e-12))

    # --- XOR ---
    p = L.init_params([2, 4, 1], 1)
    L.train(p, L.XOR_X, L.XOR_Y, lr=0.05, epochs=300)
    checks.append(("XOR: a 2-4-1 tanh network gets all 4 points right after 300 Adam epochs (a line gets at most 3)",
                   L.accuracy(p, L.XOR_X, L.XOR_Y) == 1.0))

    # --- spirals ---
    X, y = L.make_spirals(400, 0.05, 0)
    Xtr, ytr, Xte, yte = L.split(X, y, 0.25, 0)
    checks.append(("spirals data: 300 training and 100 test points, both classes present",
                   len(ytr) == 300 and len(yte) == 100 and 0.3 < ytr.mean() < 0.7))
    p = L.init_params([2, 64, 1], 0)
    h = L.train(p, Xtr, ytr, lr=0.02, epochs=600, batch=32, seed=0, Xt=Xte, yt=yte)
    checks.append((f"spirals: 2-64-1 network reaches test accuracy {h['test_acc'][-1]:.2f} (>= 0.95; measured 1.00) on unseen points",
                   h["test_acc"][-1] >= 0.95))
    checks.append((f"spirals: training accuracy {h['acc'][-1]:.3f} (>= 0.97; measured 0.993), loss fell from 0.566 to {h['loss'][-1]:.3f}",
                   h["acc"][-1] >= 0.97 and abs(h["loss"][0] - 0.566) < 0.02 and h["loss"][-1] < 0.1))
    pl = L.init_params([2, 1], 0)
    hl = L.train(pl, Xtr, ytr, lr=0.05, epochs=300, batch=32, seed=0, Xt=Xte, yt=yte)
    checks.append(("spirals: a single neuron (one straight line) reaches only 0.70 test accuracy", abs(hl["test_acc"][-1] - 0.70) < 0.05))

    # --- optimisers on moons ---
    Xm, ym = L.make_moons(300, 0.2, 1)
    res = {}
    for kind in ("sgd", "momentum", "adam"):
        pp = L.init_params([2, 16, 1], 2)
        res[kind] = L.train(pp, Xm, ym, kind=kind, lr=0.05, epochs=40, batch=32, seed=3)["loss"][-1]
    checks.append((f"after 40 epochs the loss is SGD {res['sgd']:.2f} > momentum {res['momentum']:.2f} > Adam {res['adam']:.2f} (about 0.30, 0.15, 0.06)",
                   res["sgd"] > res["momentum"] > res["adam"] and abs(res["sgd"] - 0.30) < 0.04
                   and abs(res["momentum"] - 0.15) < 0.04 and abs(res["adam"] - 0.06) < 0.03))

    # --- overfitting ---
    Xo, yo = L.make_moons(1040, 0.35, 5)
    Xs, ys, Xt, yt = Xo[:40], yo[:40], Xo[40:], yo[40:]
    gaps = {}
    for hid in (3, 100):
        pp = L.init_params([2, hid, 1], 4)
        hh = L.train(pp, Xs, ys, lr=0.01, epochs=1500, Xt=Xt, yt=yt)
        gaps[hid] = (hh["acc"][-1], hh["test_acc"][-1])
    checks.append((f"overfitting: 100 hidden neurons memorise 40 points (train {gaps[100][0]:.2f}) but test only {gaps[100][1]:.2f}; gap {gaps[100][0] - gaps[100][1]:.2f}",
                   gaps[100][0] == 1.0 and abs(gaps[100][1] - 0.80) < 0.03 and gaps[100][0] - gaps[100][1] > 0.15))
    checks.append((f"overfitting: 3 hidden neurons give train {gaps[3][0]:.3f}, test {gaps[3][1]:.2f}; a smaller gap of {gaps[3][0] - gaps[3][1]:.2f}",
                   abs(gaps[3][0] - 0.925) < 1e-9 and abs(gaps[3][1] - 0.82) < 0.03 and (gaps[3][0] - gaps[3][1]) < (gaps[100][0] - gaps[100][1])))

    # --- initialisation ---
    out = {}
    for sch in ("zeros", "const", "xavier"):
        pp = L.init_params([2, 4, 1], 1, sch)
        hh = L.train(pp, L.XOR_X, L.XOR_Y, lr=0.05, epochs=300)
        spread = float(np.abs(pp[0][0] - pp[0][0][:, [0]]).max())
        out[sch] = (hh["acc"][-1], hh["loss"][-1], spread)
    checks.append(("zero weights: nothing learns; accuracy 0.50 and loss stays at ln 2 = 0.693 after 300 epochs",
                   out["zeros"][0] == 0.5 and abs(out["zeros"][1] - np.log(2)) < 1e-9))
    checks.append(("equal weights (all 0.5): hidden neurons stay identical (spread 0) and XOR stays unsolved at 0.75",
                   out["const"][2] == 0.0 and out["const"][0] == 0.75))
    checks.append(("random weights: hidden neurons differ (spread > 1) and XOR is solved (1.00)", out["xavier"][2] > 1.0 and out["xavier"][0] == 1.0))

    # --- learning rate too high ---
    lrs = {}
    for lr in (0.05, 5.0, 50.0, 500.0):
        pp = L.init_params([2, 16, 1], 2)
        hh = L.train(pp, Xm, ym, kind="sgd", lr=lr, epochs=30, batch=32, seed=3)
        lrs[lr] = (hh["loss"][0], hh["loss"][-1], hh["acc"][-1])
    checks.append((f"lr 0.05 and 5: SGD loss falls to {lrs[0.05][1]:.2f} and {lrs[5.0][1]:.2f} (about 0.31 and 0.05)",
                   abs(lrs[0.05][1] - 0.31) < 0.05 and lrs[5.0][1] < 0.15))
    checks.append((f"lr 50 and 500: loss is {lrs[50.0][1]:.1f} and {lrs[500.0][1]:.1f} after 30 epochs, far above the start (about 0.7) and accuracy {lrs[500.0][2]:.2f}",
                   lrs[50.0][1] > 2.0 and lrs[500.0][1] > 5.0 and lrs[500.0][2] < 0.75))

    # --- activations on moons ---
    ar = {}
    for act in ("tanh", "relu", "sigmoid"):
        pp = L.init_params([2, 16, 1], 2, "he" if act == "relu" else "xavier")
        ar[act] = L.train(pp, Xm, ym, act=act, kind="adam", lr=0.05, epochs=40, batch=32, seed=3)["loss"][-1]
    checks.append((f"hidden activations after 40 Adam epochs: loss tanh {ar['tanh']:.2f}, relu {ar['relu']:.2f}, sigmoid {ar['sigmoid']:.2f} (all three learn; sigmoid slowest)",
                   ar["sigmoid"] > ar["tanh"] and ar["sigmoid"] > ar["relu"] and max(ar.values()) < 0.2))

    # --- vanishing gradients ---
    Xv, yv = L.make_moons(200, 0.2, 7)
    rat = {}
    for act, sch in (("sigmoid", "xavier"), ("tanh", "xavier"), ("relu", "he")):
        gn = L.layer_grad_norms(10, 16, act, sch, Xv, yv)
        rat[act] = gn[0] / gn[-1]
        rat[act + "_n"] = len(gn)
    checks.append((f"10 hidden layers: first-layer / last-layer gradient norm is {rat['sigmoid']:.1e} for sigmoid (vanishing), {rat['tanh']:.2f} for tanh, {rat['relu']:.2f} for relu",
                   rat["sigmoid_n"] == 11 and 5e-7 < rat["sigmoid"] < 3e-6 and 0.25 < rat["tanh"] < 0.5 and 0.25 < rat["relu"] < 0.5))
    g20 = L.layer_grad_norms(20, 16, "sigmoid", "xavier", Xv, yv)
    checks.append((f"20 hidden layers of sigmoid: first / last = {g20[0] / g20[-1]:.0e} (below 1e-10)", g20[0] / g20[-1] < 1e-10))
    z = np.linspace(-10, 10, 2001)
    checks.append(("slopes: sigmoid <= 0.25 everywhere, tanh <= 1, relu is 0 or 1",
                   float(L.ACT["sigmoid"][1](z).max()) <= 0.25 + 1e-12 and float(L.ACT["tanh"][1](z).max()) <= 1.0
                   and set(np.unique(L.ACT["relu"][1](z))) <= {0.0, 1.0}))

    # --- mini-project ---
    X, y = L.make_moons(400, 0.25, 11)
    a, b, c, d = L.split(X, y, 0.25, 0)
    accs = {}
    for hid in (2, 4, 16):
        pp = L.init_params([2, hid, 1], 0)
        hh = L.train(pp, a, b, lr=0.02, epochs=300, batch=32, seed=0, Xt=c, yt=d)
        accs[hid] = hh["test_acc"][-1]
        if hid == 16:
            cm = L.confusion(d, L.predict(pp, c))
    checks.append((f"mini-project: test accuracy {accs[2]:.2f} with 2 hidden, {accs[4]:.2f} with 4, {accs[16]:.2f} with 16 (target 0.92 needs at least 4)",
                   accs[2] < 0.92 and accs[4] >= 0.92 and accs[16] >= 0.92))
    checks.append((f"mini-project: confusion matrix for 16 hidden is {cm.tolist()} on 100 test points (3 mistakes)",
                   cm.tolist() == [[50, 2], [1, 47]] and cm.sum() == 100))
    # --- edge cases ---
    cm0 = L.confusion(np.array([0, 1]), np.array([1, 0]))
    checks.append(("edge: confusion matrix of two completely wrong answers is [[0, 1], [1, 0]]", cm0.tolist() == [[0, 1], [1, 0]]))
    checks.append(("edge: loss is finite even when the network is certain and wrong",
                   np.isfinite(L.bce(np.array([0.0, 1.0]), np.array([1.0, 0.0])))))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
